# DSC0152 - notebook v65

## Posisi terhadap v61, yang skornya 0.56312

Empat public score yang pernah tercatat:

| File | KS OOF | Public |
|---|---|---|
| v57 `aggressive_145` | 0.526988 | 0.55700 |
| v58ext `ext1.5` | 0.527511 | 0.55900 |
| tim `qc_00` | tidak terukur | 0.56171 |
| **v61 `alt_ext0.5_tm30_lean`** | 0.528624 | **0.56312** |
| v63 kandidat terbaik | **0.529867** | 0.55700 |

Dua dari tiga skor di atas 0.559 datang dari satu mekanisme: mendorong model berfitur lebih kaya menjauh dari model berfitur lebih sedikit. Itu bukan kebetulan, dan v65 menguji ulang mekanismenya di titik operasi yang sekarang.

## Aturan yang dipatuhi

Python saja, hanya data konstruksi lokal, preprocessing dan feature engineering manual, tanpa AutoML dan AutoFE, tanpa data eksternal. Agregat historis dipotong pada `decision_datetime`. Seleksi fitur dihitung per fold dari baris training fold itu saja. Factor loading dari label fold training saja. Test hanya dipakai sebagai data tanpa label untuk self-training, yang eksplisit diuji lalu ditolak. Tidak ada push ke GitHub. Semua artefak hanya di `agas/`.

In [ ]:
import os, sys, json
from pathlib import Path

import numpy as np
import pandas as pd

RUN_TRAINING = False
FOLDS = 10
FOLD_SEEDS = '42,7,13,21,5,101'
TOP_N = 250

# The notebook lives in agas/v65/notebook/, so the working directory is not
# the repository root and Path.cwd() is wrong here. Walk up from this file.
HERE = Path.cwd().resolve()
if not (HERE / 'data').is_dir():
    for cand in [HERE, *HERE.parents]:
        if (cand / 'data').is_dir() and (cand / 'agas').is_dir():
            HERE = cand
            break
ROOT = HERE
AGAS = ROOT / 'agas'
VER = AGAS / 'v65'
OUT = VER / 'output'
SHARED = AGAS / 'shared'
TRAINING = VER / 'training'
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(SHARED))
os.chdir(ROOT)
print('repo root  ', ROOT)
print('version dir', VER)

## 1. Apa yang dicoba, dan mana yang ditolak

Tiga blok fitur baru ditulis lebih dulu, lalu diuji satu per satu dengan tiga fold seed, karena satu fold seed bergerak sekitar 0.0006 AUC dari seed saja, ukuran yang sama dengan efek yang seharusnya diceap blok mana pun.

Dua dari tiga diterima, satu ditolak:

- `dataquality`: kedalaman cakupan dan fraksi missingness per tabel.
- `stress`: beban cicilan terhadap arus kas terukur, bukan income yang dideklarasikan.
- `burdirection`: arah bureauscore, utilisasi, dan delinquency. **Ditolak**, turunkan AUC dan KS di semua konfigurasi.

In [ ]:
import lightgbm as lgb, pandas as pd, numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import ks_2samp
import cv_harness as H

f, tr, te = H.load_frames(True)
X = tr.drop(columns=['application_id', 'customer_id', 'target'])
y = tr.target.astype(int).values
for c in X.columns:
    if str(X[c].dtype) == 'object':
        X[c] = X[c].astype('category')
X.shape

### Grid level seleksi fitur

v63 menemukan 200 kolom sebagai puncak, tapi itu diukur di `lr 0.02 / lambda_l2 20`. Di `lr 0.01 / lambda_l2 5` jawabannya berbeda, dan yang lebih penting adalah sebaran antar seed, bukan cuma mean.

| Level seleksi | AUC | KS | sd KS antar seed |
|---|---|---|---|
| tanpa seleksi (604 kolom) | 0.846330 | 0.525506 | 0.001454 |
| top 250 | 0.846122 | **0.526768** | **0.000342** |
| top 400 | 0.846305 | 0.525219 | 0.000217 |

Dua puluh lima ratus dipilih. Mean KS-nya paling tinggi dan sebarannya paling kecil, dan sebaran itu yang membuat bisa dipercaya.

In [ ]:
def cv(fold_seeds=(42, 7, 13), folds=5, top_n=250, lr=0.01, l2=5.0, seed=0):
    oofs, ks = [], []
    for fs in fold_seeds:
        skf = StratifiedKFold(folds, shuffle=True, random_state=fs)
        oof = np.zeros(len(y))
        for a, b in skf.split(X, y):
            p = dict(objective='binary', learning_rate=lr, num_leaves=15,
                     min_child_samples=200, feature_fraction=0.3,
                     bagging_fraction=0.8, bagging_freq=1, lambda_l2=l2,
                     max_bin=127, seed=seed, deterministic=True,
                     force_row_wise=True, n_jobs=-1, verbose=-1)
            probe = lgb.train({**p, 'feature_fraction': 1.0, 'learning_rate': 0.1},
                              lgb.Dataset(X.iloc[a], y[a]), num_boost_round=300)
            g = pd.Series(probe.feature_importance('gain'), index=X.columns)
            cols = list(g.sort_values(ascending=False).index[:top_n])
            m = lgb.train(p, lgb.Dataset(X.iloc[a][cols], y[a]),
                          num_boost_round=8000, valid_sets=[lgb.Dataset(X.iloc[b][cols], y[b])],
                          callbacks=[lgb.early_stopping(300, verbose=False)])
            oof[b] = m.predict(X.iloc[b][cols])
        oofs.append(oof)
        ks.append(ks_2samp(oof[y == 1], oof[y == 0]).statistic)
    return dict(auc=float(np.mean([roc_auc_score(y, o) for o in oofs])),
                ks=float(np.mean(ks)), ks_sd=float(np.std(ks)))

if RUN_TRAINING:
    rows = [dict(level=lv, **cv(top_n=lv)) for lv in (0, 250, 400)]
    pd.DataFrame(rows)

## 2. Self-training pada test diuji lalu ditolak

Pseudo-labeling adalah langkah transduktif standar: latih sekali, prediksi baris test, beri label keras pada baris yang paling yakin, lalu latih ulang. Paket ini harus diukur tanpa menipu diri sendiri, jadi label semu diambil hanya dari test dan skor dihitung pada validation fold yang tidak pernah masuk training di kedua tahap.

Dua setelan:

| Setelan | AUC sebelum | AUC sesudah | KS sebelum | KS sesudah |
|---|---|---|---|---|
| frac 0.20, bobot 0.5 | 0.846005 | 0.845477 | 0.526527 | 0.524218 |
| frac 0.10, bobot 0.2 | 0.846005 | 0.846012 | 0.526527 | 0.526834 |

Setelan kasar jelas merusak. Setelan halus kelihatan positif, tapi per seed delta KS-nya `+0.001346`, `-0.001870`, `+0.001448`, jadi mean positifnya datang dari dua seed yang menanggung satu seed yang lebih besar. Itu bukan peningkatan, itu pembatalan.

Penyebabnya masuk akal setelah dicek: test dan train berinterleave pada rentang tanggal yang sama, 2025-01 sampai 2025-12, dengan populasi nasabah yang sama. Tidak ada pergeseran distribusi untuk dieksploitasi, dan self-trainingyang biasanya the case ini memang tidak cocok untuk kasus ini.

In [ ]:
p = OUT / 'pseudo_frac0.1_w0.2.json'
if p.exists():
    d = json.loads(p.read_text())
    pd.DataFrame([{'fold_seed': r['fold_seed'],
                    'auc_delta': round(r['auc_b'] - r['auc_a'], 6),
                    'ks_delta': round(r['ks_b'] - r['ks_a'], 6)} for r in d['runs']])

## 3. Run utama

10 fold, enam fold seed `42,7,13,21,5,101`, empat kaki model. Seleksi 250 kolom per fold dari probe gain pada baris training fold itu saja.

Kaki `sel` memakai 250 kolom terpilih, kaki `lean` memakai seluruh kandidat, dan selisihnya adalah sumbu ekstrapolasi. Arahnya diuji dua arah, bukan diasumsikan.

In [ ]:
cmd = (f'python3 {TRAINING / "train_submission_v65.py"} --folds {FOLDS} '
       f'--fold-seeds {FOLD_SEEDS} --top-n {TOP_N} --out-prefix v65')
print(cmd)
if RUN_TRAINING:
    os.system(cmd)
else:
    print('dilewati, memakai prediksi v65 yang sudah ada')

summary = json.loads((OUT / 'summary_v65.json').read_text())
legs = pd.DataFrame([{'leg': k.replace('bag_', ''), **v}
                     for k, v in summary.items() if k.startswith('bag_')])
print('minutes', round(summary['minutes'], 1))
legs.sort_values('auc', ascending=False)

### Dua arah sumbu ekstrapolasi

v61 mendorong model terpilih menjauh dari model luas dan mendapat 0.56312. `qc_00` mendorong model berfitur kaya menjauh dari model berfitur miskin dan mendapat 0.56171. Keduanya arah yang sama hanya karena di titik operasi v61 lebih banyak fitur berarti lebih baik. Di titik operasi v65 gradiennya sudah berbalik: model 250 kolom mengalahkan model 576 kolom. Jadi kedua arah dijalankan dan hasilnya dilaporkan, bukan tandanya diasumsikan.

| Arah | Rasio | AUC | KS |
|---|---|---|---|
| `sel - lean` | 0.25 | 0.847263 | **0.528592** |
| `sel - lean` | 0.5 | 0.847071 | 0.528105 |
| `lean - sel` | 0.25 | 0.847127 | 0.525256 |
| `lean - sel` | 0.5 | 0.846896 | 0.524605 |

Arah v61 menang lagi, dan KS terbaiknya 0.528592 praktis menyamai KS OOF v61 sebesar 0.528624, padahal AUC-nya jauh lebih baik: 0.847263 melawan 0.846699. Ini kandidat pertama yang secara bersamaan cocok dengan v61 pada KS dan lebih baik pada AUC.

In [ ]:
axis = pd.DataFrame([{'axis': k.split('_r')[0].replace('axis_', ''),
                      'ratio': float(k.split('_r')[1]), **v}
                     for k, v in summary.items() if k.startswith('axis_')])
axis.sort_values('ks', ascending=False).head(6)

## 4. Pemilihan kandidat

KS OOF adalah statistik maksimum, jadi noise-nya besar. Di 30000 baris public, sebaran sampling KS sekitar 0.005. Karena itu kandidat diurutkan dengan AUC, KS dilaporkan berdampingan dengan sebaran sampling-nya, dan kandidat yang meranking hampir identik dibuang karena hanya memberi undian lain dari distribusi yang sama.

Lantai AUC dipasang sebelum deduplikasi. Tanpa itu, ekor pool terisi model lemah yang lolos justru karena meranking berbeda, dan kandidat dengan mean lebih buruk hanya bisa menurunkan nilai maksimum harapan.

In [ ]:
if RUN_TRAINING:
    os.system(f'python3 {VER / "make_submission_v65.py"}')

cs = json.loads((OUT / 'summary_v65_candidates.json').read_text())
sigma = cs['sigma_public']
print('sigma public', round(sigma, 6))
print('pool', cs['pool_size'], '-> eligible', cs['eligible'], '-> kept', cs['kept'])
cands = pd.DataFrame(cs['candidates'])
cands['ks_minus_sd'] = (cands.ks - sigma).round(6)
cands

## 5. Validasi file submission

Setiap kandidat diperiksa ulang: 30000 baris, `application_id` unik dan sama dengan `data/test_applications.csv`, probabilitas di rentang [0, 1] tanpa NaN. Jumlah file di disk juga harus sama persis dengan jumlah entri di ringkasan, supaya nomor slot tidak pernah menunjuk ke isi yang berbeda antar run.

In [ ]:
import glob

ref = set(pd.read_csv(ROOT / 'data/test_applications.csv', usecols=['application_id']).application_id)
files = sorted(glob.glob(str(OUT / 'submission_v65_*.csv')))
assert {f.split('/')[-1] for f in files} == {c['file'] for c in cs['candidates']}

rows = []
for path in files:
    d = pd.read_csv(path)
    ok = (len(d) == 30000 and d.application_id.is_unique
          and set(d.application_id) == ref
          and d.default_probability.between(0, 1).all()
          and not d.default_probability.isna().any())
    rows.append({'file': path.split('/')[-1], 'rows': len(d), 'ok': ok})
res = pd.DataFrame(rows)
print('total', len(res), '| invalid', int((~res.ok).sum()))
res

## 6. Apa yang diketahui, dan apa yang tidak

Diketahui: model v65 lebih baik dari v64 pada AUC, blend empat kaki mencapai 0.847708 yang tertinggi sepanjang proyek, dan kandidat sumbunya punya KS OOF yang praktis menyamai v61 dengan AUC yang lebih tinggi. Seleksi 250 kolom memberi sebaran antar seed paling kecil, yang membuat pilihan ini paling bisa dipercaya di antara yang diuji.

Tidak diketahui: apakah itujem translates ke public. Yang diketahui dari data adalah selisih OOF di bawah 0.002 tidak bisa dibedakan dari noise sampling, dan public leaderboard punya skor tertinggi 0.57768 sementara kita di 0.56312 pada peringkat 29. Jaraknya 0.0146, sekitar 2.75 sebaran sampling public, jadi gap itu nyata dan bukan noise: ada model yang secara verdade lebih baik, dan yang masih hilang di sini belum ditemukan.

Yang paling mungkin belumdieksploitasi:feature yang benar-benar baru. Sembilan blok yang sudah ada sudah mengaburkan sebagian besar informasi yang tersedia, dan dua blok baru di run ini menambah 0.00003 AUC saja. Feature engineering manual di dataset ini mendekati batasnya.

Rekomendasi: kirim kandidat yang saling berbeda, mulai dari `consensus_ranks` karena AUC-nya tertinggi dan modelnya paling kuat, lalu `sel-lean_r075` karena punya KS OOF tertinggi di pool. Catat public score aktual setiap file, karena itulah satu-satunya ukuran yang benar.

In [ ]:
r = os.system('python3 agas/v64/verify_all_scripts.py')
print('verify exit code', r)